# 玩家统计数据分析

本 Notebook 用于分析 PlayerStats，输入玩家名称后展示：
- 核心指标：VPIP、PFR、WTP、AGG
- Preflop 参数分析
- Postflop 参数分析

In [3]:
# 环境初始化
import sys
from pathlib import Path

# notebook 位于 src/bayes_poker/analysis/，往上 3 层是项目根目录
project_root = Path.cwd().parent.parent.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"src 路径已添加: {src_path}")

项目根目录: /home
src 路径已添加: /home/src


In [4]:
# 导入依赖
import pandas as pd
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import seaborn as sns
from bayes_poker.player_metrics.enums import (
    ActionType,
    Position,
    PreflopPotType,
    Street,
    TableType,
)
from bayes_poker.player_metrics.models import ActionStats, PlayerStats
from bayes_poker.player_metrics.params import PostFlopParams, PreFlopParams
from bayes_poker.player_metrics.builder import (
    calculate_aggression,
    calculate_pfr,
    calculate_total_hands,
    calculate_wtp,
)
from bayes_poker.storage.player_stats_repository import PlayerStatsRepository
from bayes_poker.player_metrics.enums import TableType
from bayes_poker.storage.player_stats_repository import PlayerStatsRepository
print("依赖导入成功!")

依赖导入成功!


## 1. 配置与数据加载

In [11]:
# ======== 配置区域 ========

# 数据库路径（相对于项目根目录）
DB_PATH = Path("/home/autumn/bayes_poker/data/database/player_stats.db")

# 玩家名称（输入你要分析的玩家ID）
PLAYER_NAME = "scofield5656"  # <-- 修改这里

# 桌型：TableType.HEADS_UP (2人) 或 TableType.SIX_MAX (6人)
TABLE_TYPE = TableType.SIX_MAX  # <-- 修改这里

print(f"数据库路径: {DB_PATH}")
print(f"玩家名称: {PLAYER_NAME}")
print(f"桌型: {TABLE_TYPE.name}")

数据库路径: /home/autumn/bayes_poker/data/database/player_stats.db
玩家名称: scofield5656
桌型: SIX_MAX


In [12]:
# 加载玩家数据
def load_player_stats(db_path: Path, player_name: str, table_type: TableType) -> PlayerStats | None:
    """从数据库加载玩家统计数据。"""
    if not db_path.exists():
        print(f"❌ 数据库文件不存在: {db_path}")
        return None
    
    with PlayerStatsRepository(db_path) as repo:
        stats = repo.get(player_name, table_type)
        if stats is None:
            print(f"❌ 未找到玩家: {player_name} (桌型: {table_type.name})")
            # 列出可用玩家
            all_stats = repo.get_all(table_type)
            if all_stats:
                print(f"\n可用玩家列表 ({len(all_stats)} 人):")
                for s in sorted(all_stats, key=lambda x: x.vpip.total, reverse=True)[:20]:
                    print(f"  - {s.player_name} (手数: {s.vpip.total})")
            return None
        return stats

player_stats = load_player_stats(DB_PATH, PLAYER_NAME, TABLE_TYPE)
if player_stats:
    print(f"✅ 成功加载玩家数据: {player_stats.player_name}")

✅ 成功加载玩家数据: scofield5656


## 2. 核心指标 (VPIP / PFR / WTP / AGG)

In [5]:
def format_stat(positive: int, total: int) -> str:
    """格式化统计值为百分比字符串。"""
    if total == 0:
        return "N/A (0)"
    pct = positive / total * 100
    return f"{pct:.1f}% ({positive}/{total})"

def display_core_stats(stats: PlayerStats) -> pd.DataFrame:
    """显示核心统计指标。"""
    # 计算各项指标
    total_hands = calculate_total_hands(stats)
    vpip_pct = stats.vpip.to_float() * 100
    
    pfr_pos, pfr_total = calculate_pfr(stats)
    pfr_pct = pfr_pos / pfr_total * 100 if pfr_total > 0 else 0
    
    wtp_pos, wtp_total = calculate_wtp(stats)
    wtp_pct = wtp_pos / wtp_total * 100 if wtp_total > 0 else 0
    
    agg_pos, agg_total = calculate_aggression(stats)
    agg_pct = agg_pos / agg_total * 100 if agg_total > 0 else 0
    
    data = {
        "指标": ["总手数", "VPIP", "PFR", "WTP", "AGG"],
        "值": [
            f"{total_hands}",
            format_stat(stats.vpip.positive, stats.vpip.total),
            format_stat(pfr_pos, pfr_total),
            format_stat(wtp_pos, wtp_total),
            format_stat(agg_pos, agg_total),
        ],
        "百分比": [
            "-",
            f"{vpip_pct:.1f}%",
            f"{pfr_pct:.1f}%",
            f"{wtp_pct:.1f}%",
            f"{agg_pct:.1f}%",
        ],
        "说明": [
            "统计样本数",
            "自愿投入底池比例（Voluntarily Put money In Pot）",
            "翻前加注比例（Pre-Flop Raise）",
            "面对下注时跟注/加注比例（Went To Pot）",
            "激进度：加注数 / (加注数 + 跟注数)",
        ],
    }
    return pd.DataFrame(data)

if player_stats:
    print(f"\n📊 {player_stats.player_name} 核心统计指标\n")
    df_core = display_core_stats(player_stats)
    display(df_core.style.set_properties(**{"text-align": "left"}))


📊 scofield5656 核心统计指标



,指标,值,百分比,说明
0,总手数,2481,-,统计样本数
1,VPIP,22.2% (550/2481),22.2%,自愿投入底池比例（Voluntarily Put money In Pot）
2,PFR,19.8% (491/2481),19.8%,翻前加注比例（Pre-Flop Raise）
3,WTP,49.2% (87/177),49.2%,面对下注时跟注/加注比例（Went To Pot）
4,AGG,76.6% (193/252),76.6%,激进度：加注数 / (加注数 + 跟注数)


## 3. Preflop 参数分析

In [6]:
def analyze_preflop_stats(stats: PlayerStats) -> pd.DataFrame:
    """分析 Preflop 统计数据。"""
    all_params = PreFlopParams.get_all_params(stats.table_type)
    
    rows = []
    for i, params in enumerate(all_params):
        action_stats = stats.preflop_stats[i]
        total = action_stats.total_samples()
        if total == 0:
            continue  # 跳过无样本的参数组合
        
        row = {
            "索引": i,
            "位置": params.position.name,
            "加注数": params.num_raises,
            "跟注数": params.num_callers,
            "前动作": params.previous_action.name,
            "IP": "是" if params.in_position_on_flop else "否",
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df.sort_values("样本数", ascending=False)

if player_stats:
    print(f"\n🃏 {player_stats.player_name} Preflop 参数分析\n")
    df_preflop = analyze_preflop_stats(player_stats)
    print(f"共 {len(df_preflop)} 种有样本的参数组合\n")
    display(df_preflop.head(20))


🃏 scofield5656 Preflop 参数分析

共 23 种有样本的参数组合



,索引,位置,加注数,跟注数,前动作,IP,样本数,加注%,跟注%,弃牌%
9,10,UTG,0,0,FOLD,否,423,19.1%,0.0%,80.9%
10,15,HJ,0,0,FOLD,否,299,21.7%,0.0%,78.3%
13,20,CO,0,0,FOLD,否,270,28.5%,0.0%,71.5%
6,7,BIG_BLIND,1,0,FOLD,否,226,15.0%,23.9%,61.1%
2,2,SMALL_BLIND,1,0,FOLD,否,208,15.4%,0.5%,84.1%
18,25,BUTTON,0,0,FOLD,否,194,41.8%,0.0%,58.2%
20,27,BUTTON,1,0,FOLD,否,171,10.5%,0.0%,89.5%
0,0,SMALL_BLIND,0,0,FOLD,否,147,43.5%,0.0%,56.5%
15,22,CO,1,0,FOLD,否,142,11.3%,0.0%,88.7%
12,17,HJ,1,0,FOLD,否,70,5.7%,0.0%,94.3%


In [7]:
def preflop_by_position(stats: PlayerStats) -> pd.DataFrame:
    """按位置汇总 Preflop 统计。"""
    all_params = PreFlopParams.get_all_params(stats.table_type)
    
    position_stats: dict[str, ActionStats] = {}
    
    for i, params in enumerate(all_params):
        # 只统计 previous_action == FOLD 的（即首次行动）
        if params.previous_action != ActionType.FOLD:
            continue
        
        pos_name = params.position.name
        if pos_name not in position_stats:
            position_stats[pos_name] = ActionStats()
        position_stats[pos_name].append(stats.preflop_stats[i])
    
    rows = []
    for pos_name, action_stats in position_stats.items():
        total = action_stats.total_samples()
        if total == 0:
            continue
        rows.append({
            "位置": pos_name,
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        })
    
    # 按位置顺序排序
    pos_order = {"SMALL_BLIND": 0, "BIG_BLIND": 1, "UTG": 2, "HJ": 3, "CO": 4, "BUTTON": 5}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_order"] = df["位置"].map(lambda x: pos_order.get(x, 99))
        df = df.sort_values("_order").drop(columns=["_order"])
    return df

if player_stats:
    print(f"\n📍 按位置汇总 Preflop 行动\n")
    df_by_pos = preflop_by_position(player_stats)
    display(df_by_pos)


📍 按位置汇总 Preflop 行动



,位置,样本数,加注%,跟注%,弃牌%
0,SMALL_BLIND,417,23.5%,0.2%,76.3%
1,BIG_BLIND,339,13.3%,21.5%,65.2%
2,UTG,423,19.1%,0.0%,80.9%
3,HJ,377,18.3%,0.0%,81.7%
4,CO,429,21.7%,0.0%,78.3%
5,BUTTON,411,25.5%,0.0%,74.5%


## 4. Postflop 参数分析

In [8]:
def analyze_postflop_stats(stats: PlayerStats) -> pd.DataFrame:
    """分析 Postflop 统计数据。"""
    all_params = PostFlopParams.get_all_params(stats.table_type)
    
    rows = []
    for i, params in enumerate(all_params):
        action_stats = stats.postflop_stats[i]
        total = action_stats.total_samples()
        if total == 0:
            continue
        
        row = {
            "索引": i,
            "街": params.street.name,
            "轮次": params.round,
            "前动作": params.prev_action.name,
            "下注数": params.num_bets,
            "IP": "是" if params.in_position else "否",
            "玩家数": params.num_players,
            "底池类型": params.preflop_pot_type.name,
            "PF攻击者": "是" if params.is_preflop_aggressor else "否",
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df.sort_values("样本数", ascending=False)

if player_stats:
    print(f"\n🎯 {player_stats.player_name} Postflop 参数分析\n")
    df_postflop = analyze_postflop_stats(player_stats)
    print(f"共 {len(df_postflop)} 种有样本的参数组合\n")
    display(df_postflop.head(20))


🎯 scofield5656 Postflop 参数分析

共 122 种有样本的参数组合



,索引,街,轮次,前动作,下注数,IP,玩家数,底池类型,PF攻击者,样本数,加注%,跟注%,弃牌%
58,864,FLOP,0,RAISE,0,否,2,THREE_BET_PLUS,否,73,56.2%,43.8%,0.0%
15,230,FLOP,0,CALL,0,是,2,LIMPED,是,41,7.3%,92.7%,0.0%
60,866,FLOP,0,RAISE,0,是,2,THREE_BET_PLUS,否,32,25.0%,75.0%,0.0%
20,282,FLOP,1,CHECK,1,是,2,LIMPED,是,26,23.1%,23.1%,53.8%
71,960,TURN,0,CHECK,0,否,2,THREE_BET_PLUS,否,24,29.2%,70.8%,0.0%
66,936,TURN,0,RAISE,0,否,2,THREE_BET_PLUS,否,21,66.7%,33.3%,0.0%
65,930,FLOP,1,CHECK,1,是,2,THREE_BET_PLUS,否,18,16.7%,38.9%,44.4%
95,1082,FLOP,0,RAISE,0,是,2,THREE_BET_PLUS,是,16,50.0%,50.0%,0.0%
59,865,FLOP,0,RAISE,0,否,3,THREE_BET_PLUS,否,15,13.3%,86.7%,0.0%
86,1032,RIVER,0,CHECK,0,否,2,THREE_BET_PLUS,否,15,60.0%,40.0%,0.0%


In [9]:
def postflop_by_street(stats: PlayerStats) -> pd.DataFrame:
    """按街汇总 Postflop 统计。"""
    all_params = PostFlopParams.get_all_params(stats.table_type)
    
    street_stats: dict[str, ActionStats] = {}
    
    for i, params in enumerate(all_params):
        street_name = params.street.name
        if street_name not in street_stats:
            street_stats[street_name] = ActionStats()
        street_stats[street_name].append(stats.postflop_stats[i])
    
    rows = []
    for street_name, action_stats in street_stats.items():
        total = action_stats.total_samples()
        if total == 0:
            continue
        rows.append({
            "街": street_name,
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        })
    
    street_order = {"FLOP": 0, "TURN": 1, "RIVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_order"] = df["街"].map(lambda x: street_order.get(x, 99))
        df = df.sort_values("_order").drop(columns=["_order"])
    return df

if player_stats:
    print(f"\n🎴 按街汇总 Postflop 行动\n")
    df_by_street = postflop_by_street(player_stats)
    display(df_by_street)


🎴 按街汇总 Postflop 行动



,街,样本数,加注%,跟注%,弃牌%
0,FLOP,323,29.7%,56.7%,13.6%
1,TURN,190,32.1%,51.6%,16.3%
2,RIVER,115,31.3%,55.7%,13.0%


In [10]:
def postflop_by_position_and_street(stats: PlayerStats) -> pd.DataFrame:
    """按位置和街汇总 Postflop 统计。"""
    all_params = PostFlopParams.get_all_params(stats.table_type)
    
    # key: (in_position, street)
    grouped_stats: dict[tuple[bool, str], ActionStats] = {}
    
    for i, params in enumerate(all_params):
        key = (params.in_position, params.street.name)
        if key not in grouped_stats:
            grouped_stats[key] = ActionStats()
        grouped_stats[key].append(stats.postflop_stats[i])
    
    rows = []
    for (in_pos, street), action_stats in grouped_stats.items():
        total = action_stats.total_samples()
        if total == 0:
            continue
        rows.append({
            "位置": "IP" if in_pos else "OOP",
            "街": street,
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        })
    
    street_order = {"FLOP": 0, "TURN": 1, "RIVER": 2}
    pos_order = {"OOP": 0, "IP": 1}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_street_order"] = df["街"].map(lambda x: street_order.get(x, 99))
        df["_pos_order"] = df["位置"].map(lambda x: pos_order.get(x, 99))
        df = df.sort_values(["_street_order", "_pos_order"]).drop(columns=["_street_order", "_pos_order"])
    return df

if player_stats:
    print(f"\n📊 按位置和街汇总 Postflop 行动\n")
    df_pos_street = postflop_by_position_and_street(player_stats)
    display(df_pos_street)


📊 按位置和街汇总 Postflop 行动



,位置,街,样本数,加注%,跟注%,弃牌%
0,OOP,FLOP,154,40.3%,51.3%,8.4%
1,IP,FLOP,169,20.1%,61.5%,18.3%
2,OOP,TURN,102,33.3%,51.0%,15.7%
3,IP,TURN,88,30.7%,52.3%,17.0%
4,OOP,RIVER,64,32.8%,54.7%,12.5%
5,IP,RIVER,51,29.4%,56.9%,13.7%


## 5. 下注尺度分析

In [11]:
def analyze_bet_sizing(stats: PlayerStats) -> pd.DataFrame:
    """分析下注尺度分布。"""
    all_postflop = PostFlopParams.get_all_params(stats.table_type)
    
    # 按街汇总下注尺度
    street_sizing: dict[str, dict[str, int]] = {}
    
    for i, params in enumerate(all_postflop):
        action_stats = stats.postflop_stats[i]
        street = params.street.name
        
        if street not in street_sizing:
            street_sizing[street] = {
                "0-40%": 0,
                "40-80%": 0,
                "80-120%": 0,
                ">120%": 0,
            }
        
        street_sizing[street]["0-40%"] += action_stats.bet_0_40
        street_sizing[street]["40-80%"] += action_stats.bet_40_80
        street_sizing[street]["80-120%"] += action_stats.bet_80_120
        street_sizing[street][">120%"] += action_stats.bet_over_120
    
    rows = []
    for street, sizing in street_sizing.items():
        total = sum(sizing.values())
        if total == 0:
            continue
        rows.append({
            "街": street,
            "总下注数": total,
            "0-40%": f"{sizing['0-40%']} ({sizing['0-40%']/total*100:.1f}%)",
            "40-80%": f"{sizing['40-80%']} ({sizing['40-80%']/total*100:.1f}%)",
            "80-120%": f"{sizing['80-120%']} ({sizing['80-120%']/total*100:.1f}%)",
            ">120%": f"{sizing['>120%']} ({sizing['>120%']/total*100:.1f}%)",
        })
    
    street_order = {"FLOP": 0, "TURN": 1, "RIVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_order"] = df["街"].map(lambda x: street_order.get(x, 99))
        df = df.sort_values("_order").drop(columns=["_order"])
    return df

if player_stats:
    print(f"\n💰 {player_stats.player_name} 下注尺度分析\n")
    df_sizing = analyze_bet_sizing(player_stats)
    display(df_sizing)


💰 scofield5656 下注尺度分析



,街,总下注数,0-40%,40-80%,80-120%,>120%
0,FLOP,96,45 (46.9%),34 (35.4%),13 (13.5%),4 (4.2%)
1,TURN,61,28 (45.9%),18 (29.5%),6 (9.8%),9 (14.8%)
2,RIVER,36,8 (22.2%),15 (41.7%),7 (19.4%),6 (16.7%)


## 6. 底池类型分析

In [12]:
def analyze_by_pot_type(stats: PlayerStats) -> pd.DataFrame:
    """按底池类型分析 Postflop 行为。"""
    all_params = PostFlopParams.get_all_params(stats.table_type)
    
    pot_type_stats: dict[str, ActionStats] = {}
    
    for i, params in enumerate(all_params):
        pot_type = params.preflop_pot_type.name
        if pot_type not in pot_type_stats:
            pot_type_stats[pot_type] = ActionStats()
        pot_type_stats[pot_type].append(stats.postflop_stats[i])
    
    rows = []
    for pot_type, action_stats in pot_type_stats.items():
        total = action_stats.total_samples()
        if total == 0:
            continue
        rows.append({
            "底池类型": pot_type,
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        })
    
    pot_order = {"LIMPED": 0, "SINGLE_RAISED": 1, "THREE_BET_PLUS": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_order"] = df["底池类型"].map(lambda x: pot_order.get(x, 99))
        df = df.sort_values("_order").drop(columns=["_order"])
    return df

if player_stats:
    print(f"\n🏆 按底池类型分析 Postflop 行为\n")
    df_pot_type = analyze_by_pot_type(player_stats)
    display(df_pot_type)


🏆 按底池类型分析 Postflop 行为



,底池类型,样本数,加注%,跟注%,弃牌%
0,LIMPED,197,18.3%,62.4%,19.3%
1,SINGLE_RAISED,27,29.6%,59.3%,11.1%
2,THREE_BET_PLUS,404,36.9%,51.0%,12.1%


In [13]:
def analyze_aggressor_vs_caller(stats: PlayerStats) -> pd.DataFrame:
    """分析作为 Preflop 攻击者 vs 跟注者的 Postflop 行为差异。"""
    all_params = PostFlopParams.get_all_params(stats.table_type)
    
    aggressor_stats = ActionStats()
    caller_stats = ActionStats()
    
    for i, params in enumerate(all_params):
        if params.is_preflop_aggressor:
            aggressor_stats.append(stats.postflop_stats[i])
        else:
            caller_stats.append(stats.postflop_stats[i])
    
    rows = []
    for role, action_stats in [("PF攻击者", aggressor_stats), ("PF跟注者", caller_stats)]:
        total = action_stats.total_samples()
        if total == 0:
            continue
        rows.append({
            "角色": role,
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        })
    
    return pd.DataFrame(rows)

if player_stats:
    print(f"\n⚔️ Preflop 攻击者 vs 跟注者 Postflop 行为对比\n")
    df_agg_vs_caller = analyze_aggressor_vs_caller(player_stats)
    display(df_agg_vs_caller)


⚔️ Preflop 攻击者 vs 跟注者 Postflop 行为对比



,角色,样本数,加注%,跟注%,弃牌%
0,PF攻击者,244,28.3%,54.5%,17.2%
1,PF跟注者,384,32.3%,55.2%,12.5%


## 7. 数据导出

In [ ]:
# 如需导出分析结果到 CSV，取消下面的注释

# if player_stats:
#     output_dir = project_root / "analysis" / "outputs"
#     output_dir.mkdir(parents=True, exist_ok=True)
#     
#     df_core.to_csv(output_dir / f"{PLAYER_NAME}_core_stats.csv", index=False)
#     df_preflop.to_csv(output_dir / f"{PLAYER_NAME}_preflop.csv", index=False)
#     df_postflop.to_csv(output_dir / f"{PLAYER_NAME}_postflop.csv", index=False)
#     
#     print(f"✅ 数据已导出到: {output_dir}")

## 8. 低样本玩家：平滑 vs 非平滑

这一节固定展示一个低样本玩家，用来观察玩家池后验平滑对频率估计的收缩效果。

- 示例玩家: `AndrewYuen`
- 手牌数: `20`
- 平滑强度: `20.0`

这个玩家从 `20-50` 手的候选区间中筛出，优先选择平滑前后节点分布变化较明显、且 preflop / postflop 都有可读样本的示例。

In [13]:
from bayes_poker.player_metrics.analysis_helpers import (
    build_core_stats_comparison,
    build_node_delta_table,
)

LOW_SAMPLE_PLAYER_NAME = "AndrewYuen"
LOW_SAMPLE_PLAYER_HANDS = 20
POOL_PRIOR_STRENGTH = 20.0


def load_player_stats_comparison(
    db_path: Path,
    player_name: str,
    table_type: TableType,
    pool_prior_strength: float,
) -> tuple[PlayerStats | None, PlayerStats | None]:
    """加载同一玩家的两份统计引用。

    Args:
        db_path: SQLite 数据库路径。
        player_name: 玩家名称。
        table_type: 桌型。
        pool_prior_strength: 兼容旧调用保留, 当前未使用。

    Returns:
        `(stats, stats)` 二元组。任一读取失败时返回 `(None, None)`。
    """
    if not db_path.exists():
        print(f"❌ 数据库文件不存在: {db_path}")
        return None, None

    with PlayerStatsRepository(db_path) as repo:
        raw_stats = repo.get(player_name, table_type)
        smoothed_stats = raw_stats

    if raw_stats is None or smoothed_stats is None:
        print(f"❌ 未找到玩家: {player_name} (桌型: {table_type.name})")
        return None, None

    return raw_stats, smoothed_stats


In [14]:
raw_low_sample_stats, smoothed_low_sample_stats = load_player_stats_comparison(
    Path(DB_PATH),
    LOW_SAMPLE_PLAYER_NAME,
    TABLE_TYPE,
    POOL_PRIOR_STRENGTH,
)

if raw_low_sample_stats and smoothed_low_sample_stats:
    print(
        f"\n🔬 低样本示例玩家: {LOW_SAMPLE_PLAYER_NAME} "
        f"(原始手数: {LOW_SAMPLE_PLAYER_HANDS}, k={POOL_PRIOR_STRENGTH})\n"
    )
    display(build_core_stats_comparison(raw_low_sample_stats, smoothed_low_sample_stats))

    print("\n📍 Preflop 变化最大的节点\n")
    display(
        build_node_delta_table(
            raw_low_sample_stats,
            smoothed_low_sample_stats,
            scope="preflop",
            top_n=10,
        )
    )

    print("\n🎯 Postflop 变化最大的节点\n")
    display(
        build_node_delta_table(
            raw_low_sample_stats,
            smoothed_low_sample_stats,
            scope="postflop",
            top_n=10,
        )
    )



🔬 低样本示例玩家: AndrewYuen (原始手数: 20, k=20.0)



,指标,原始值,平滑值,原始百分比,平滑百分比
0,总手数,20,20,-,-
1,VPIP,11/20,11/20,55.0%,55.0%
2,PFR,7/19,114.438/619,36.8%,18.5%
3,WTP,9/11,9816.52/17291,81.8%,56.8%
4,AGG,12/15,7976.31/13738.1,80.0%,58.1%



📍 Preflop 变化最大的节点



,索引,节点描述,原始样本数,平滑样本数,原始加注%,平滑加注%,原始跟注%,平滑跟注%,原始弃牌%,平滑弃牌%,L1差异
0,27,BUTTON | raises=1 | callers=0 | prev=FOLD | fl...,1,21.0,0.0,7.6,100.0,14.6,0.0,77.8,170.7868
1,2,SMALL_BLIND | raises=1 | callers=0 | prev=FOLD...,1,21.0,0.0,7.5,100.0,15.1,0.0,77.4,169.8763
2,7,BIG_BLIND | raises=1 | callers=0 | prev=FOLD |...,2,22.0,0.0,6.5,100.0,33.9,0.0,59.6,132.2923
3,25,BUTTON | raises=0 | callers=0 | prev=FOLD | fl...,1,21.0,100.0,38.4,0.0,0.0,0.0,61.6,123.2420
4,0,SMALL_BLIND | raises=0 | callers=0 | prev=FOLD...,1,21.0,100.0,39.8,0.0,4.4,0.0,55.8,120.3805
5,10,UTG | raises=0 | callers=0 | prev=FOLD | flop_...,2,22.0,50.0,21.1,0.0,0.0,50.0,78.9,57.7504
6,15,HJ | raises=0 | callers=0 | prev=FOLD | flop_ip=否,2,22.0,50.0,23.0,0.0,0.0,50.0,77.0,53.9356
7,22,CO | raises=1 | callers=0 | prev=FOLD | flop_ip=否,3,23.0,33.3,10.5,0.0,7.1,66.7,82.4,45.6080
8,20,CO | raises=0 | callers=0 | prev=FOLD | flop_ip=否,4,24.0,50.0,28.8,0.0,0.0,50.0,71.2,42.3082
9,17,HJ | raises=1 | callers=0 | prev=FOLD | flop_ip=否,1,21.0,0.0,6.0,0.0,7.0,100.0,87.0,25.9661



🎯 Postflop 变化最大的节点



,索引,节点描述,原始样本数,平滑样本数,原始加注%,平滑加注%,原始跟注%,平滑跟注%,原始弃牌%,平滑弃牌%,L1差异
0,1002,TURN | round=1 | prev=CHECK | bets=1 | IP=是 | ...,1,21.0,100.0,10.3,0.0,37.6,0.0,52.2,179.4839
1,908,FLOP | round=1 | prev=RAISE | bets=2 | IP=否 | ...,1,21.0,100.0,12.4,0.0,47.5,0.0,40.1,175.2613
2,282,FLOP | round=1 | prev=CHECK | bets=1 | IP=是 | ...,1,21.0,100.0,12.4,0.0,43.5,0.0,44.1,175.2444
3,1180,TURN | round=0 | prev=CHECK | bets=1 | IP=否 | ...,1,21.0,100.0,12.8,0.0,43.9,0.0,43.3,174.4848
4,304,TURN | round=0 | prev=CALL | bets=1 | IP=否 | p...,1,21.0,100.0,15.4,0.0,50.1,0.0,34.5,169.1033
5,229,FLOP | round=0 | prev=CALL | bets=0 | IP=否 | p...,1,21.0,100.0,16.0,0.0,84.0,0.0,0.0,168.0569
6,1228,RIVER | round=0 | prev=RAISE | bets=1 | IP=否 |...,1,21.0,100.0,16.0,0.0,46.0,0.0,37.9,167.9290
7,1080,FLOP | round=0 | prev=RAISE | bets=0 | IP=否 | ...,1,21.0,0.0,70.5,100.0,29.5,0.0,0.0,140.9651
8,426,RIVER | round=1 | prev=CHECK | bets=1 | IP=是 |...,1,21.0,0.0,4.3,100.0,37.0,0.0,58.7,125.9942
9,386,RIVER | round=0 | prev=CHECK | bets=0 | IP=是 |...,1,21.0,100.0,38.9,0.0,61.1,0.0,0.0,122.1386


## 9. 当前配置玩家：平滑 vs 非平滑

这一节直接复用上面的 `PLAYER_NAME` 配置。默认当前 notebook 里配置的是 `scofield5656`，因此这里会展示这个玩家在开启和关闭玩家池后验平滑时的差异。

In [15]:
configured_raw_stats, configured_smoothed_stats = load_player_stats_comparison(
    Path(DB_PATH),
    PLAYER_NAME,
    TABLE_TYPE,
    POOL_PRIOR_STRENGTH,
)

if configured_raw_stats and configured_smoothed_stats:
    print(
        f"\n🧪 当前配置玩家: {PLAYER_NAME} "
        f"(总手数: {calculate_total_hands(configured_raw_stats)}, k={POOL_PRIOR_STRENGTH})\n"
    )
    display(build_core_stats_comparison(configured_raw_stats, configured_smoothed_stats))

    print("\n📍 Preflop 变化最大的节点\n")
    display(
        build_node_delta_table(
            configured_raw_stats,
            configured_smoothed_stats,
            scope="preflop",
            top_n=10,
        )
    )

    print("\n🎯 Postflop 变化最大的节点\n")
    display(
        build_node_delta_table(
            configured_raw_stats,
            configured_smoothed_stats,
            scope="postflop",
            top_n=10,
        )
    )



🧪 当前配置玩家: scofield5656 (总手数: 2481, k=20.0)



,指标,原始值,平滑值,原始百分比,平滑百分比
0,总手数,2481,2481,-,-
1,VPIP,550/2481,550/2481,22.2%,22.2%
2,PFR,491/2396,598.438/2996,20.5%,20.0%
3,WTP,87/177,9894.52/17457,49.2%,56.7%
4,AGG,193/252,8157.31/13975.1,76.6%,58.4%



📍 Preflop 变化最大的节点



,索引,节点描述,原始样本数,平滑样本数,原始加注%,平滑加注%,原始跟注%,平滑跟注%,原始弃牌%,平滑弃牌%,L1差异
0,21,CO | raises=0 | callers=1 | prev=FOLD | flop_ip=否,7,27.0,0.0,12.3,0.0,3.6,100.0,84.1,31.8903
1,16,HJ | raises=0 | callers=1 | prev=FOLD | flop_ip=否,8,28.0,0.0,10.7,0.0,3.3,100.0,86.0,28.0170
2,1,SMALL_BLIND | raises=0 | callers=1 | prev=FOLD...,10,30.0,10.0,13.5,0.0,10.2,90.0,76.3,27.3862
3,23,CO | raises=1 | callers=1 | prev=FOLD | flop_ip=否,4,24.0,0.0,5.0,0.0,7.4,100.0,87.6,24.7436
4,6,BIG_BLIND | raises=0 | callers=1 | prev=FOLD |...,23,43.0,34.8,26.2,65.2,73.8,0.0,0.0,17.1841
5,3,SMALL_BLIND | raises=1 | callers=1 | prev=FOLD...,31,51.0,3.2,4.3,0.0,4.8,96.8,90.9,11.6762
6,28,BUTTON | raises=1 | callers=1 | prev=FOLD | fl...,20,40.0,10.0,8.2,0.0,5.7,90.0,86.1,11.3665
7,8,BIG_BLIND | raises=1 | callers=1 | prev=FOLD |...,49,69.0,4.1,4.6,8.2,13.3,87.8,82.1,11.2228
8,26,BUTTON | raises=0 | callers=1 | prev=FOLD | fl...,11,31.0,27.3,21.9,0.0,4.4,72.7,73.7,10.8079
9,24,CO | raises=2 | callers=0 | prev=FOLD | flop_ip=否,6,26.0,0.0,1.6,0.0,2.9,100.0,95.5,9.0124



🎯 Postflop 变化最大的节点



,索引,节点描述,原始样本数,平滑样本数,原始加注%,平滑加注%,原始跟注%,平滑跟注%,原始弃牌%,平滑弃牌%,L1差异
0,1012,RIVER | round=0 | prev=RAISE | bets=1 | IP=否 |...,1,21.0,100.0,19.3,0.0,36.5,0.0,44.2,161.3920
1,364,RIVER | round=0 | prev=RAISE | bets=1 | IP=否 |...,1,21.0,100.0,21.3,0.0,38.2,0.0,40.5,157.4222
2,1156,TURN | round=0 | prev=RAISE | bets=1 | IP=否 | ...,1,21.0,100.0,25.3,0.0,44.0,0.0,30.6,149.3076
3,292,TURN | round=0 | prev=RAISE | bets=1 | IP=否 | ...,1,21.0,100.0,25.8,0.0,45.3,0.0,28.9,148.4110
4,388,RIVER | round=0 | prev=CHECK | bets=1 | IP=否 |...,1,21.0,0.0,8.6,100.0,33.1,0.0,58.3,133.8220
5,1168,TURN | round=0 | prev=CALL | bets=1 | IP=否 | p...,1,21.0,0.0,15.1,0.0,51.3,100.0,33.6,132.7764
6,1196,TURN | round=1 | prev=RAISE | bets=2 | IP=否 | ...,1,21.0,0.0,8.8,0.0,54.3,100.0,36.9,126.1916
7,520,TURN | round=0 | prev=CALL | bets=1 | IP=否 | p...,1,21.0,0.0,12.3,0.0,47.7,100.0,40.0,119.9051
8,372,RIVER | round=0 | prev=CALL | bets=0 | IP=否 | ...,1,21.0,100.0,43.3,0.0,56.7,0.0,0.0,113.4360
9,578,RIVER | round=0 | prev=RAISE | bets=0 | IP=是 |...,1,21.0,0.0,54.9,100.0,45.1,0.0,0.0,109.7017
